In [ ]:
#things to toggle on the interface

#parametrize:
# the room dimensions and wall curvature
# gravity
# air type and density
# shape of the ball
# material properties of the ball
# initial conditions


In [19]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel, Field

class Ball(BaseModel):
    """a 3d ball with material and state"""
    radius: float
    mass: float
    color: str = "blue"
    #material properties
    elasticity: float
    density: float
    dilation: float = 0.0

    #initial conditions
    start_position: list[float]
    start_velocity: list[float]
    current_position: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    current_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    acceleration: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    force: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))

    class Config:
        arbitrary_types_allowed = True

    def initialize_state(self) -> None:
        self.current_position = np.array(self.start_position, dtype=float)
        self.current_velocity = np.array(self.start_velocity, dtype=float)
        self.acceleration = np.zeros(3, dtype=float)
        self.force = np.zeros(3, dtype=float)


medium_types = Literal["air", "helium", "nitrogen"]
class BounceEnvironment(BaseModel):
    """environment parameters for the room"""

    #room features
    room_dimensions: list[float]
    gravity: float
    gravity_direction: list[float] = [0.0, -1.0, 0.0]
    linear_drag: float = 0.0
    quadratic_drag: float = 0.0
    medium_type: medium_types = "air"
    restitution: float = 0.9
    wall_curvature: float = 0.0
    air_density: float = 1.225

    class Config:
        arbitrary_types_allowed = True


class Simulation(BaseModel):
    """run a physics simulation for multiple balls"""
    balls: list[Ball]
    environment: BounceEnvironment
    time_step: float
    total_time_steps: int
    positions: list[np.ndarray] = Field(default_factory=list)
    velocities: list[np.ndarray] = Field(default_factory=list)
    occupancy_grid: np.ndarray | None = None
    occupancy_history: list[np.ndarray] = Field(default_factory=list)

    class Config:
        arbitrary_types_allowed = True

    def _gravity_vector(self) -> np.ndarray:
        g_dir = np.array(self.environment.gravity_direction, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            return np.zeros(3, dtype=float)
        return (g_dir / g_norm) * self.environment.gravity

    def calculate_forces(self) -> None:
        g_vec = self._gravity_vector()
        for ball in self.balls:
            ball.force = ball.mass * g_vec

            v = ball.current_velocity
            speed = np.linalg.norm(v)
            if self.environment.linear_drag != 0.0:
                ball.force += -self.environment.linear_drag * v
            if self.environment.quadratic_drag != 0.0 and speed > 0.0:
                ball.force += -self.environment.quadratic_drag * speed * v

    def update_acceleration(self) -> None:
        for ball in self.balls:
            ball.acceleration = ball.force / ball.mass

    def update_velocity(self) -> None:
        for ball in self.balls:
            ball.current_velocity = (
                ball.current_velocity + ball.acceleration * self.time_step
            )

    def update_position(self) -> None:
        for ball in self.balls:
            ball.current_position = (
                ball.current_position + ball.current_velocity * self.time_step
            )

    def update_deformation(self) -> None:
        for ball in self.balls:
            speed = np.linalg.norm(ball.current_velocity)
            ball.dilation = ball.dilation + speed * self.time_step

    def bounce_off_walls(self, ball: Ball) -> None:
        dims = np.array(self.environment.room_dimensions, dtype=float)
        pos = ball.current_position
        vel = ball.current_velocity
        r = ball.radius
        restitution = self.environment.restitution

        for axis in range(3):
            min_bound = r
            max_bound = dims[axis] - r
            if pos[axis] < min_bound:
                pos[axis] = min_bound
                vel[axis] = -vel[axis] * restitution
            elif pos[axis] > max_bound:
                pos[axis] = max_bound
                vel[axis] = -vel[axis] * restitution

        ball.current_position = pos
        ball.current_velocity = vel

    def handle_ball_collisions(self) -> None:
        restitution = self.environment.restitution
        count = len(self.balls)

        for i in range(count):
            for j in range(i + 1, count):
                b1 = self.balls[i]
                b2 = self.balls[j]
                delta = b2.current_position - b1.current_position
                dist = np.linalg.norm(delta)
                min_dist = b1.radius + b2.radius

                if dist >= min_dist:
                    continue

                if dist == 0.0:
                    normal = np.array([1.0, 0.0, 0.0], dtype=float)
                else:
                    normal = delta / dist

                overlap = min_dist - dist
                b1.current_position -= normal * (overlap / 2.0)
                b2.current_position += normal * (overlap / 2.0)

                rel_vel = b2.current_velocity - b1.current_velocity
                vel_along_normal = np.dot(rel_vel, normal)
                if vel_along_normal > 0.0:
                    continue

                inv_m1 = 0.0 if b1.mass == 0.0 else 1.0 / b1.mass
                inv_m2 = 0.0 if b2.mass == 0.0 else 1.0 / b2.mass
                impulse_mag = (-(1.0 + restitution) * vel_along_normal) / (inv_m1 + inv_m2)
                impulse = impulse_mag * normal

                b1.current_velocity -= impulse * inv_m1
                b2.current_velocity += impulse * inv_m2

    def _snapshot(self) -> tuple[np.ndarray, np.ndarray]:
        positions = np.stack([b.current_position for b in self.balls], axis=0)
        velocities = np.stack([b.current_velocity for b in self.balls], axis=0)
        return positions, velocities

    def init_grid(self, resolution: int = 20) -> None:
        self.occupancy_grid = np.zeros((resolution, resolution, resolution), dtype=float)

    def update_occupancy(self) -> None:
        if self.occupancy_grid is None:
            return
        dims = np.array(self.environment.room_dimensions, dtype=float)
        res = self.occupancy_grid.shape[0]
        for ball in self.balls:
            idx = ((ball.current_position / dims) * (res - 1)).astype(int)
            idx = np.clip(idx, 0, res - 1)
            self.occupancy_grid[tuple(idx)] += 1.0

    def step(self) -> None:
        self.calculate_forces()
        self.update_acceleration()
        self.update_velocity()
        self.update_position()
        self.update_deformation()
        for ball in self.balls:
            self.bounce_off_walls(ball)
        self.handle_ball_collisions()
        positions, velocities = self._snapshot()
        self.positions.append(positions)
        self.velocities.append(velocities)
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())

    def simulate(self) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for _ in range(self.total_time_steps):
            self.step()


/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_39002/2951054624.py:6: PydanticDeprecatedSince20:

Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_39002/2951054624.py:35: PydanticDeprecatedSince20:

Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_39002/2951054624.py:53: PydanticDeprecatedSince20:

Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/



In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ast


def _random_non_overlapping_positions(radii, room_dims, max_tries=5000):
    rng = np.random.default_rng()
    positions = []

    for r in radii:
        mins = np.array([r, r, r], dtype=float)
        maxs = np.array(room_dims, dtype=float) - r
        placed = False

        for _ in range(max_tries):
            candidate = rng.uniform(mins, maxs)
            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} without overlap")
    return positions


def _initial_positions(radii, room_dims, layout="random", max_tries=5000):
    """generate initial positions for different layouts while avoiding overlap"""
    rng = np.random.default_rng()
    dims = np.array(room_dims, dtype=float)

    if layout == "random":
        return _random_non_overlapping_positions(radii, room_dims, max_tries=max_tries)

    positions = []
    center = dims / 2.0

    for idx, r in enumerate(radii):
        placed = False
        for _ in range(max_tries):
            if layout == "cluster_center":
                spread = dims.min() * 0.2
                candidate = center + rng.normal(scale=spread, size=3)
            elif layout == "cluster_corner":
                base = np.array([r, r, r], dtype=float)
                spread = dims.min() * 0.2
                candidate = base + rng.normal(scale=spread, size=3)
            elif layout == "line":
                t = idx / max(1, len(radii) - 1)
                candidate = np.array([
                    r + t * (dims[0] - 2 * r),
                    dims[1] * 0.5,
                    dims[2] * 0.5,
                ], dtype=float)
            elif layout == "high_drop":
                candidate = np.array([
                    rng.uniform(r, dims[0] - r),
                    rng.uniform(r, dims[1] - r),
                    dims[2] * 0.8,
                ], dtype=float)
            else:
                candidate = rng.uniform([r, r, r], dims - r)

            if np.any(candidate < r) or np.any(candidate > dims - r):
                continue

            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} for layout {layout}")

    return positions


def _build_balls(count, radius_range, room_dims, speed, density, layout="random", elasticity=0.5, color="blue"):
    rng = np.random.default_rng()
    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)

    positions = _initial_positions(radii, room_dims, layout=layout)

    balls = []
    for i in range(count):
        r = radii[i]
        m = density * (4.0 / 3.0) * np.pi * (r ** 3)

        direction = rng.normal(size=3)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0.0:
            direction = np.array([1.0, 0.0, 0.0], dtype=float)
            direction_norm = 1.0
        velocity = (direction / direction_norm) * speed

        balls.append(
            Ball(
                radius=r,
                mass=m,
                color=color,
                start_position=positions[i].tolist(),
                start_velocity=velocity.tolist(),
                elasticity=elasticity,
                density=density,
            )
        )
    return balls


def _compute_scalar_fields(positions, velocities, balls, env, mode="velocity"):
    values_per_frame = []

    if mode == "velocity":
        for v in velocities:
            values_per_frame.append(np.linalg.norm(v, axis=1))
    elif mode == "potential_energy":
        g_dir = np.array(env.gravity_direction, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            g_dir = np.array([0.0, 0.0, -1.0], dtype=float)
            g_norm = 1.0
        unit_g = g_dir / g_norm
        masses = np.array([b.mass for b in balls], dtype=float)

        for pos in positions:
            height = -np.dot(pos, unit_g)
            pe = masses * env.gravity * height
            values_per_frame.append(pe)
    else:
        for pos in positions:
            values_per_frame.append(np.zeros(pos.shape[0], dtype=float))

    all_vals = np.concatenate(values_per_frame) if values_per_frame else np.array([0.0])
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if vmax <= vmin:
        vmax = vmin + 1.0

    return values_per_frame, vmin, vmax


def _plot_with_density(sim, balls, env, room_dims, frame_stride=1, color_mode="velocity"):
    positions = sim.positions
    velocities = sim.velocities
    occupancy_history = sim.occupancy_history
    dims = np.array(room_dims, dtype=float)

    scene_config = dict(
        xaxis=dict(range=[0, dims[0]], autorange=False),
        yaxis=dict(range=[0, dims[1]], autorange=False),
        zaxis=dict(range=[0, dims[2]], autorange=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=dims[1] / dims[0], z=dims[2] / dims[0]),
    )

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6
    else:
        r_max = radii.max()
        if r_max <= 0.0:
            marker_sizes = 6
        else:
            marker_sizes = 4.0 + (radii / r_max) * 8.0

    scalar_values, vmin, vmax = _compute_scalar_fields(positions, velocities, balls, env, mode=color_mode)

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "scene", "colspan": 2}, None],
            [{"type": "xy"}, {"type": "scene"}],
        ],
        subplot_titles=(
            "3d simulation",
            "",
            "cumulative occupancy (xy plane)",
            "occupancy surface (kde-like)",
        ),
    )

    initial = positions[0]
    initial_vals = scalar_values[0]
    initial_heat = occupancy_history[0] if occupancy_history else np.zeros((10, 10))

    fig.add_trace(
        go.Scatter3d(
            x=initial[:, 0],
            y=initial[:, 1],
            z=initial[:, 2],
            mode="markers",
            marker=dict(
                size=marker_sizes,
                color=initial_vals,
                colorscale="Viridis",
                cmin=vmin,
                cmax=vmax,
                opacity=0.8,
                colorbar=dict(title=color_mode),
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(z=initial_heat, colorscale="Hot", showscale=False),
        row=2,
        col=1,
    )

    x_idx = np.linspace(0, dims[0], initial_heat.shape[1])
    y_idx = np.linspace(0, dims[1], initial_heat.shape[0])
    xx, yy = np.meshgrid(x_idx, y_idx)

    fig.add_trace(
        go.Surface(z=initial_heat, x=xx, y=yy, colorscale="Hot", showscale=False, opacity=0.9),
        row=2,
        col=2,
    )

    frames = []
    for t in range(0, len(positions), frame_stride):
        pos = positions[t]
        vals = scalar_values[t]
        heat = occupancy_history[t] if t < len(occupancy_history) else initial_heat
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=pos[:, 0],
                        y=pos[:, 1],
                        z=pos[:, 2],
                        mode="markers",
                        marker=dict(
                            size=marker_sizes,
                            color=vals,
                            colorscale="Viridis",
                            cmin=vmin,
                            cmax=vmax,
                            opacity=0.8,
                        ),
                    ),
                    go.Heatmap(z=heat, colorscale="Hot", showscale=False),
                    go.Surface(z=heat, x=xx, y=yy, colorscale="Hot", showscale=False, opacity=0.9),
                ],
                name=str(t),
                traces=[0, 1, 2],
            )
        )

    fig.update_layout(
        scene=scene_config,
        uirevision="constant_view",
        margin=dict(l=0, r=0, b=0, t=30),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True},
                        ],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
    )

    fig.frames = frames

    return fig


ball_count = widgets.IntSlider(value=50, min=1, max=1000, step=1, description="balls")
radius_range = widgets.FloatRangeSlider(
    value=[0.3, 0.8], min=0.1, max=2.0, step=0.1, description="radius range"
)
ball_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
ball_speed = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="speed")

room_dims = widgets.Text(
    value="[10.0, 10.0, 10.0]",
    description="room [x,y,z]",
)

gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="bounciness")

material = widgets.Dropdown(
    options=["rubber", "steel", "foam"],
    value="rubber",
    description="material",
)

color_mode = widgets.Dropdown(
    options=[
        ("velocity", "velocity"),
        ("potential energy", "potential_energy"),
    ],
    value="velocity",
    description="color by",
)

init_layout = widgets.Dropdown(
    options=[
        ("random", "random"),
        ("cluster center", "cluster_center"),
        ("cluster corner", "cluster_corner"),
        ("line", "line"),
        ("high drop", "high_drop"),
    ],
    value="random",
    description="layout",
)

run_button = widgets.Button(description="generate sim")
output = widgets.Output()


def _run_simulation(_):
    output.clear_output(wait=True)
    with output:
        try:
            dims_val = ast.literal_eval(room_dims.value)
            dims = [float(v) for v in dims_val]
            if len(dims) != 3:
                raise ValueError
        except Exception:
            dims = [10.0, 10.0, 10.0]

        material_props = {
            "rubber": {"elasticity": 0.9, "color": "red"},
            "steel": {"elasticity": 0.6, "color": "gray"},
            "foam": {"elasticity": 0.8, "color": "orange"},
        }
        props = material_props[material.value]
        elasticity = props["elasticity"]
        color = props["color"]

        env = BounceEnvironment(
            room_dimensions=dims,
            gravity=gravity.value,
            gravity_direction=[0.0, 0.0, -1.0],
            linear_drag=0.0,
            quadratic_drag=0.0,
            restitution=restitution.value,
        )
        balls = _build_balls(
            ball_count.value,
            radius_range.value,
            dims,
            ball_speed.value,
            ball_density.value,
            layout=init_layout.value,
            elasticity=elasticity,
            color=color,
        )
        sim = Simulation(
            balls=balls,
            environment=env,
            time_step=0.02,
            total_time_steps=2000,
        )
        sim.simulate()
        frame_stride = max(1, len(sim.positions) // 300)
        fig = _plot_with_density(
            sim,
            balls,
            env,
            dims,
            frame_stride=frame_stride,
            color_mode=color_mode.value,
        )
        display(fig)


run_button.on_click(_run_simulation)

controls = widgets.VBox(
    [
        ball_count,
        radius_range,
        ball_density,
        ball_speed,
        room_dims,
        gravity,
        restitution,
        material,
        color_mode,
        init_layout,
        run_button,
    ]
)

ui = widgets.HBox([controls, output])
display(ui)



In [ ]:


#things to add potentially:
#energy/entropoy of the balls by color codes
#how much potential energy is lost in the collision

#what is the gridcell that is the most touched throughout the simulation?
#monte carlo simulation of the grid space that is most entered by the balls